# Caltrain On-Time Performance Heatmap (2025)

This notebook visualizes the on-time percentage for each day of 2025 using a calendar heatmap.
Each square represents a day, colored by the on-time percentage for that day.

Data is loaded directly from the database and processed to calculate delays.

In [3]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Add project root to path for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# Database path
DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
print(f"Database path: {DB_PATH}")
print(f"Database exists: {os.path.exists(DB_PATH)}")

Database path: d:\caltrain-tracker\data\caltrain_lat_long.db
Database exists: True


## Load and Process Data

We'll load raw data from the database and process it to calculate delays.
This logic is repurposed from `data_processing.py`.

In [4]:
# Import utility functions from the project
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_raw_train_data(year=2025):
    """Load raw train location data from the database for a specific year."""
    conn = sqlite3.connect(DB_PATH)
    
    query = f"""
    SELECT 
        trip_id,
        stop_id,
        vehicle_lat,
        vehicle_lon,
        timestamp
    FROM train_locations
    WHERE timestamp LIKE '{year}%'
    """
    
    try:
        df = pd.read_sql_query(query, conn)
        print(f"Loaded {len(df):,} raw records from database for {year}")
        
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            df = df.dropna(subset=['timestamp'])
            print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
        
        return df
    finally:
        conn.close()

def load_gtfs_data():
    """Load GTFS static data (stops and stop times)."""
    gtfs_path = os.path.join(project_root, 'gtfs_data')
    
    stops_df = pd.read_csv(os.path.join(gtfs_path, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    
    stop_times_df = pd.read_csv(os.path.join(gtfs_path, 'stop_times.txt'))
    
    print(f"Loaded {len(stops_df)} stops and {len(stop_times_df):,} stop times from GTFS")
    return stops_df, stop_times_df

def process_arrival_data(raw_df, stops_df, stop_times_df):
    """Process raw train data to calculate delays (repurposed from data_processing.py)."""
    if raw_df.empty or stops_df.empty or stop_times_df.empty:
        print("WARNING: One or more input DataFrames are empty")
        return pd.DataFrame()
    
    print("Processing arrival data...")
    
    # Ensure consistent data types - convert to string first, then to int
    # This handles cases where trip_id might be stored as text in database
    raw_df = raw_df.copy()
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    stops_df = stops_df.copy()
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    
    stop_times_df = stop_times_df.copy()
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    # Drop rows with NaN in key columns
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    print(f"After type conversion: {len(raw_df):,} records")
    
    # Merge datasets
    df2 = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], 
                   on=['trip_id', 'stop_id'], how='inner')
    df2 = pd.merge(df2, stops_df[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
                   on=['stop_id'], how='inner')
    
    print(f"Merged data: {len(df2):,} records")
    
    # Calculate distance between train and stop
    df2['distance'] = df2.apply(lambda row: haversine(
        row['vehicle_lat'], row['vehicle_lon'], row['stop_lat'], row['stop_lon']
    ), axis=1)
    
    df2['date'] = df2['timestamp'].dt.date
    df2['arrival_time'] = df2['arrival_time'].apply(normalize_time)
    
    # Find minimum distance for each trip-stop-date
    min_distances = df2.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged_df = pd.merge(df2, min_distances, on=['trip_id', 'stop_id', 'date', 'distance'])
    
    # Get closest approach time
    arrival_times = merged_df.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    arrival_times = arrival_times[['trip_id', 'stop_id', 'date', 'timestamp', 'arrival_time', 'stop_name']]
    arrival_times.rename(columns={'timestamp': 'actual_arrival_time'}, inplace=True)
    
    print(f"Calculated {len(arrival_times):,} arrival events")
    
    # Calculate delay
    arrival_times['delay_minutes'] = arrival_times.apply(
        lambda row: calculate_time_difference(row['arrival_time'], row['actual_arrival_time']), 
        axis=1
    )
    
    # Clean unrealistic delays
    arrival_times.loc[arrival_times.delay_minutes > 500, 'delay_minutes'] = 0.0
    arrival_times.loc[arrival_times.delay_minutes < -100, 'delay_minutes'] = 0.0
    
    # Determine severity
    arrival_times['delay_severity'] = 'On Time'
    arrival_times.loc[(arrival_times.delay_minutes > 4) & (arrival_times.delay_minutes <= 15), 'delay_severity'] = 'Minor'
    arrival_times.loc[arrival_times.delay_minutes > 15, 'delay_severity'] = 'Major'
    arrival_times.loc[arrival_times.delay_minutes < 0, 'delay_minutes'] = 0
    
    print(f"Processing complete!")
    return arrival_times

# Load and process data
print("Loading raw train data...")
raw_df = load_raw_train_data(year=2025)

print("\nLoading GTFS schedule data...")
stops_df, stop_times_df = load_gtfs_data()

print("\nProcessing arrivals and calculating delays...")
df = process_arrival_data(raw_df, stops_df, stop_times_df)

print(f"\nFinal: {len(df):,} arrival events with delay data")
print(f"\nDelay severity counts:")
print(df['delay_severity'].value_counts())
df.head()

Loading raw train data...
Loaded 1,526,864 raw records from database for 2025
Date range: 2025-08-10 11:14:40 to 2025-12-28 10:49:54

Loading GTFS schedule data...
Loaded 64 stops and 3,694 stop times from GTFS

Processing arrivals and calculating delays...
Processing arrival data...
After type conversion: 1,524,149 records
Merged data: 1,523,542 records
Calculated 243,075 arrival events
Processing complete!

Final: 243,075 arrival events with delay data

Delay severity counts:
delay_severity
On Time    224212
Minor       16035
Major        2828
Name: count, dtype: int64


,trip_id,stop_id,date,actual_arrival_time,arrival_time,stop_name,delay_minutes,delay_severity
0,101,70021,2025-08-11,2025-08-11 05:55:48,5:55:00,22nd Street Caltrain Station Northbound,0.800000,On Time
1,101,70021,2025-08-12,2025-08-12 05:56:00,5:55:00,22nd Street Caltrain Station Northbound,1.000000,On Time
2,101,70021,2025-08-13,2025-08-13 05:55:58,5:55:00,22nd Street Caltrain Station Northbound,0.966667,On Time
3,101,70021,2025-08-14,2025-08-14 05:55:58,5:55:00,22nd Street Caltrain Station Northbound,0.966667,On Time
4,101,70021,2025-08-15,2025-08-15 05:58:46,5:55:00,22nd Street Caltrain Station Northbound,3.766667,On Time


## Calculate Daily Statistics

In [5]:
def calculate_daily_on_time_percentage(df):
    """Calculate on-time percentage for each day."""
    if df.empty:
        return pd.DataFrame()
    
    # Calculate percentage on-time per day
    daily_summary = df.groupby('date')['delay_severity'].apply(
        lambda x: (x == 'On Time').mean() * 100
    ).reset_index()
    daily_summary.columns = ['date', 'on_time_pct']
    
    # Get counts
    daily_counts = df.groupby('date').size().reset_index(name='count')
    daily_summary = daily_summary.merge(daily_counts, on='date')
    daily_summary['date'] = pd.to_datetime(daily_summary['date'])
    
    print(f"Calculated stats for {len(daily_summary)} days")
    print(f"Average on-time: {daily_summary['on_time_pct'].mean():.1f}%")
    
    return daily_summary

daily_stats = calculate_daily_on_time_percentage(df)
daily_stats.head(10)

Calculated stats for 141 days
Average on-time: 92.1%


,date,on_time_pct,count
0,2025-08-10,86.889693,1106
1,2025-08-11,94.401244,1929
2,2025-08-12,92.756644,1919
3,2025-08-13,92.291774,1933
4,2025-08-14,91.455205,1931
5,2025-08-15,91.791430,1937
6,2025-08-16,89.413448,1398
7,2025-08-17,93.366619,1402
8,2025-08-18,75.487013,1848
9,2025-08-19,94.573643,1935


## Create Calendar Heatmap

In [6]:
def prepare_calendar_data(daily_stats, year=2025):
    """Prepare data for calendar heatmap."""
    start_date = pd.Timestamp(f'{year}-01-01')
    end_date = pd.Timestamp(f'{year}-12-31')
    all_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    complete_df = pd.DataFrame({'date': all_dates})
    
    if not daily_stats.empty:
        daily_stats = daily_stats.copy()
        daily_stats['date'] = pd.to_datetime(daily_stats['date'])
        complete_df = complete_df.merge(daily_stats, on='date', how='left')
    else:
        complete_df['on_time_pct'] = np.nan
        complete_df['count'] = 0
    
    complete_df['day_of_week'] = complete_df['date'].dt.dayofweek
    first_day = complete_df['date'].iloc[0]
    complete_df['week_id'] = ((complete_df['date'] - first_day).dt.days + first_day.dayofweek) // 7
    
    return complete_df

calendar_data = prepare_calendar_data(daily_stats, year=2025)
print(f"Calendar: {len(calendar_data)} days total")
print(f"Days with data: {calendar_data['on_time_pct'].notna().sum()}")

Calendar: 365 days total
Days with data: 141


In [7]:
def create_calendar_heatmap(calendar_data, year=2025):
    """Create GitHub-style calendar heatmap."""
    if calendar_data is None or calendar_data.empty:
        print("No data available")
        return None
    
    num_weeks = calendar_data['week_id'].max() + 1
    heatmap_values = np.full((7, num_weeks), np.nan)
    hover_texts = [['' for _ in range(num_weeks)] for _ in range(7)]
    
    for _, row in calendar_data.iterrows():
        week_idx = int(row['week_id'])
        day_idx = int(row['day_of_week'])
        
        if pd.notna(row['on_time_pct']):
            heatmap_values[day_idx, week_idx] = row['on_time_pct']
            hover_texts[day_idx][week_idx] = (
                f"<b>{row['date'].strftime('%A, %b %d, %Y')}</b><br>"
                f"On-Time: <b>{row['on_time_pct']:.1f}%</b><br>"
                f"Arrivals: {int(row['count']) if pd.notna(row['count']) else 0}"
            )
        else:
            heatmap_values[day_idx, week_idx] = -1
            hover_texts[day_idx][week_idx] = f"<b>{row['date'].strftime('%A, %b %d, %Y')}</b><br>No data"
    
    day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    
    # Month labels
    month_positions = []
    month_labels = []
    current_month = None
    
    for _, row in calendar_data.iterrows():
        month = row['date'].strftime('%b')
        if month != current_month:
            month_positions.append(row['week_id'])
            month_labels.append(month)
            current_month = month
    
    colorscale = [
        [0.0, '#161b22'],
        [0.01, '#161b22'],
        [0.02, '#da3633'],
        [0.25, '#f85149'],
        [0.50, '#d29922'],
        [0.75, '#3fb950'],
        [1.0, '#238636']
    ]
    
    fig = go.Figure(data=go.Heatmap(
        z=heatmap_values,
        x=list(range(num_weeks)),
        y=day_labels,
        text=hover_texts,
        hoverinfo='text',
        colorscale=colorscale,
        zmin=-1,
        zmax=100,
        colorbar=dict(
            title=dict(text='On-Time %', font=dict(color='#c9d1d9')),
            thickness=15,
            len=0.6,
            ticksuffix='%',
            tickvals=[0, 25, 50, 75, 100],
            tickfont=dict(color='#c9d1d9'),
        ),
        xgap=3,
        ygap=3,
    ))
    
    valid_data = calendar_data[calendar_data['on_time_pct'].notna()]
    if not valid_data.empty:
        avg_on_time = valid_data['on_time_pct'].mean()
        total_arrivals = valid_data['count'].sum()
        days_tracked = len(valid_data)
        subtitle = f"Average: {avg_on_time:.1f}% on-time | {days_tracked} days | {int(total_arrivals):,} arrivals"
    else:
        subtitle = "No data"
    
    fig.update_layout(
        title=dict(
            text=f'🚆 Caltrain On-Time Performance {year}<br><span style="font-size:14px;color:#8b949e">{subtitle}</span>',
            font=dict(size=24, color='#c9d1d9'),
            x=0.5,
            xanchor='center'
        ),
        xaxis=dict(
            tickmode='array',
            tickvals=month_positions,
            ticktext=month_labels,
            tickfont=dict(size=12, color='#8b949e'),
            side='top',
            showgrid=False,
        ),
        yaxis=dict(
            tickfont=dict(size=11, color='#8b949e'),
            showgrid=False,
            autorange='reversed',
        ),
        plot_bgcolor='#0d1117',
        paper_bgcolor='#0d1117',
        height=280,
        width=1100,
        margin=dict(l=50, r=120, t=100, b=30),
    )
    
    return fig

fig = create_calendar_heatmap(calendar_data, year=2025)
if fig:
    fig.show()

## Summary Statistics

In [8]:
def display_summary_stats(daily_stats):
    """Display summary statistics."""
    if daily_stats.empty:
        print("No data")
        return
    
    valid_stats = daily_stats[daily_stats['on_time_pct'].notna()]
    
    print("=" * 60)
    print("🚆 CALTRAIN ON-TIME PERFORMANCE - 2025")
    print("=" * 60)
    print(f"\n📅 Days with data: {len(valid_stats)}")
    print(f"🚉 Total arrivals: {valid_stats['count'].sum():,}")
    print(f"\n📊 Overall on-time: {valid_stats['on_time_pct'].mean():.1f}%")
    
    best_idx = valid_stats['on_time_pct'].idxmax()
    worst_idx = valid_stats['on_time_pct'].idxmin()
    
    print(f"\n✅ Best: {valid_stats.loc[best_idx, 'date'].strftime('%b %d')} ({valid_stats['on_time_pct'].max():.1f}%)")
    print(f"❌ Worst: {valid_stats.loc[worst_idx, 'date'].strftime('%b %d')} ({valid_stats['on_time_pct'].min():.1f}%)")
    
    # Day of week
    valid_stats = valid_stats.copy()
    valid_stats['day_of_week'] = valid_stats['date'].dt.day_name()
    dow_avg = valid_stats.groupby('day_of_week')['on_time_pct'].mean()
    
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    dow_avg = dow_avg.reindex([d for d in day_order if d in dow_avg.index])
    
    print("\n📆 By Day of Week:")
    print("-" * 40)
    for day, pct in dow_avg.items():
        bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
        emoji = "🟢" if pct >= 80 else "🟡" if pct >= 60 else "🔴"
        print(f"{emoji} {day:10} {pct:5.1f}% |{bar}|")
    
    # Monthly
    valid_stats['month'] = valid_stats['date'].dt.strftime('%B')
    monthly_avg = valid_stats.groupby('month')['on_time_pct'].mean()
    
    month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                   'July', 'August', 'September', 'October', 'November', 'December']
    monthly_avg = monthly_avg.reindex([m for m in month_order if m in monthly_avg.index])
    
    print("\n📅 By Month:")
    print("-" * 40)
    for month, pct in monthly_avg.items():
        bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
        emoji = "🟢" if pct >= 80 else "🟡" if pct >= 60 else "🔴"
        print(f"{emoji} {month:10} {pct:5.1f}% |{bar}|")

display_summary_stats(daily_stats)

🚆 CALTRAIN ON-TIME PERFORMANCE - 2025

📅 Days with data: 141
🚉 Total arrivals: 243,075

📊 Overall on-time: 92.1%

✅ Best: Nov 27 (99.0%)
❌ Worst: Oct 04 (63.7%)

📆 By Day of Week:
----------------------------------------
🟢 Monday      92.7% |██████████████████░░|
🟢 Tuesday     92.7% |██████████████████░░|
🟢 Wednesday   93.3% |██████████████████░░|
🟢 Thursday    92.5% |██████████████████░░|
🟢 Friday      91.7% |██████████████████░░|
🟢 Saturday    90.7% |██████████████████░░|
🟢 Sunday      90.9% |██████████████████░░|

📅 By Month:
----------------------------------------
🟢 August      91.4% |██████████████████░░|
🟢 September   94.1% |██████████████████░░|
🟢 October     90.4% |██████████████████░░|
🟢 November    92.4% |██████████████████░░|
🟢 December    91.9% |██████████████████░░|


## Save Heatmap

In [9]:
if fig:
    os.makedirs(os.path.join(project_root, 'static', 'plots'), exist_ok=True)
    output_path = os.path.join(project_root, 'static', 'plots', 'on_time_heatmap_2025.html')
    fig.write_html(output_path, include_plotlyjs='cdn')
    print(f"✅ Saved to: {output_path}")

✅ Saved to: d:\caltrain-tracker\static\plots\on_time_heatmap_2025.html
